# Appendix. 05 - Gravity modeling and IPF

This notebook estimates comparable gravity models for the expanded HTS and pre-expanded LCS OD flows.

The workflow:

1. Loads the overall HTS and LCS OD matrices created in `01_HTS_preprocessing_and_expansion.ipynb` and `02_LCS_preprocessing_and_averaging.ipynb`.
2. Computes OD-level flow-weighted mean travel time from the corresponding processed trip tables.
3. Retains **interzonal OD pairs with non-zero flows in both HTS and LCS**.
4. Adds the origin and destination mass variables used in the gravity specification.
5. Estimates the log-linear gravity model separately for HTS and LCS.
6. Generates unconstrained predictions and IPF-adjusted predictions.
7. Evaluates both predictions using **CPC and R²**.

CPC is reported as a ratio from 0 to 1, consistent with the manuscript tables.

## Inputs

Outputs created by the preceding notebooks:

- `1_data_preprocessing/HTS_trip_weighted.csv`
- `1_data_preprocessing/LCS_trip_averaged.csv`
- `2_data_OD/OD_HTS.csv`
- `2_data_OD/OD_LCS.csv`

Additional gravity-model input:

- `0_data/Gravity_input_TAZ-level.csv`
  - `행정동코드`: analysis-zone ID
  - `총생활인구수`: zone mass used as `m_o` and `m_d`

`03_Mismatches_in_OD_Trip_Distributions.ipynb` and `04_Data_construction_and_regression.ipynb` are not inputs to this notebook.

In [ ]:
from pathlib import Path

from IPython.display import display
import numpy as np
import pandas as pd

# =========================================================
# Project paths
# =========================================================
BASE_DIR = Path.cwd()

RAW_DIR = BASE_DIR / "0_data"
PREP_DIR = BASE_DIR / "1_data_preprocessing"
OD_DATA_DIR = BASE_DIR / "2_data_OD"
GRAVITY_RESULT_DIR = BASE_DIR / "6_Result_Gravity_Modeling"

GRAVITY_RESULT_DIR.mkdir(parents=True, exist_ok=True)

# Inputs created in 01 and 02
HTS_TRIP_FILE = PREP_DIR / "HTS_trip_weighted.csv"
LCS_TRIP_FILE = PREP_DIR / "LCS_trip_averaged.csv"
HTS_OD_FILE = OD_DATA_DIR / "OD_HTS.csv"
LCS_OD_FILE = OD_DATA_DIR / "OD_LCS.csv"

# Additional input used for the origin and destination mass terms
ZONE_MASS_FILE = RAW_DIR / "Gravity_input_TAZ-level.csv"

# Outputs
GRAVITY_INPUT_SUMMARY_FILE = (
    GRAVITY_RESULT_DIR / "gravity_model_input_summary.csv"
)
GRAVITY_PARAMETER_FILE = (
    GRAVITY_RESULT_DIR / "gravity_model_parameters.csv"
)
GRAVITY_PERFORMANCE_FILE = (
    GRAVITY_RESULT_DIR / "gravity_model_performance.csv"
)
GRAVITY_PREDICTION_FILE = (
    GRAVITY_RESULT_DIR / "gravity_model_predictions.csv"
)

# IPF settings
IPF_MAX_ITER = 1000
IPF_TOL = 1e-10
IPF_EPS = 1e-12

## 1. Define helper functions

In [ ]:
def read_csv_flexible(path: Path) -> pd.DataFrame:
    """Read a project CSV file using common encodings."""
    for encoding in ["utf-8-sig", "utf-8", "cp949"]:
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue

    raise ValueError(
        f"Unable to read file with supported encodings: {path}"
    )


def standardize_zone_id(series: pd.Series) -> pd.Series:
    """Convert zone IDs to stable string keys."""
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    ).astype("Int64")

    if numeric.isna().any():
        raise ValueError(
            f"Zone ID contains {numeric.isna().sum():,} "
            "missing or invalid values."
        )

    return numeric.astype("string")


def prepare_od_flow_table(
    df: pd.DataFrame,
    flow_name: str,
) -> pd.DataFrame:
    """Standardize an overall OD flow table created in 01 or 02."""
    out = df.copy()
    out.columns = out.columns.astype(str).str.strip()

    required = ["origin_id", "destination_id", "trips"]
    missing = [col for col in required if col not in out.columns]

    if missing:
        raise KeyError(
            f"Missing OD columns: {missing}. "
            f"Available columns: {out.columns.tolist()}"
        )

    out["origin_id"] = standardize_zone_id(out["origin_id"])
    out["destination_id"] = standardize_zone_id(out["destination_id"])
    out[flow_name] = pd.to_numeric(out["trips"], errors="coerce")

    if out[flow_name].isna().any():
        raise ValueError(
            f"{flow_name} contains missing or invalid values."
        )

    out = (
        out.groupby(
            ["origin_id", "destination_id"],
            as_index=False,
            observed=True,
        )[flow_name]
        .sum()
    )

    return out


def aggregate_od_travel_time(
    trip_df: pd.DataFrame,
    output_col: str,
) -> pd.DataFrame:
    """Compute OD-level flow-weighted mean travel time."""
    df = trip_df.copy()
    df.columns = df.columns.astype(str).str.strip()

    required = [
        "origin_id",
        "destination_id",
        "travel_time_min",
        "trips",
    ]
    missing = [col for col in required if col not in df.columns]

    if missing:
        raise KeyError(
            f"Missing trip columns: {missing}. "
            f"Available columns: {df.columns.tolist()}"
        )

    df["origin_id"] = standardize_zone_id(df["origin_id"])
    df["destination_id"] = standardize_zone_id(df["destination_id"])
    df["travel_time_min"] = pd.to_numeric(
        df["travel_time_min"],
        errors="coerce",
    )
    df["trips"] = pd.to_numeric(
        df["trips"],
        errors="coerce",
    )

    df = df.loc[
        df["travel_time_min"].notna()
        & df["trips"].notna()
        & df["trips"].gt(0)
    ].copy()

    df["weighted_time"] = (
        df["travel_time_min"] * df["trips"]
    )

    od_time = (
        df.groupby(
            ["origin_id", "destination_id"],
            as_index=False,
            observed=True,
        )
        .agg(
            weighted_time_sum=("weighted_time", "sum"),
            flow_sum=("trips", "sum"),
        )
    )

    od_time[output_col] = (
        od_time["weighted_time_sum"] / od_time["flow_sum"]
    )

    return od_time[
        ["origin_id", "destination_id", output_col]
    ].copy()


def load_zone_mass(path: Path) -> pd.DataFrame:
    """Load the zone-level mass variable used in the gravity model."""
    mass = read_csv_flexible(path)
    mass.columns = mass.columns.astype(str).str.strip()

    required = ["행정동코드", "총생활인구수"]
    missing = [col for col in required if col not in mass.columns]

    if missing:
        raise KeyError(
            f"Missing zone-mass columns: {missing}. "
            f"Available columns: {mass.columns.tolist()}"
        )

    mass = mass[required].copy()
    mass["zone_id"] = standardize_zone_id(mass["행정동코드"])
    mass["zone_mass"] = pd.to_numeric(
        mass["총생활인구수"],
        errors="coerce",
    )

    mass = mass[["zone_id", "zone_mass"]].drop_duplicates()

    if mass["zone_id"].duplicated().any():
        raise ValueError(
            "The zone-mass file contains multiple values for one zone."
        )

    if mass["zone_mass"].isna().any() or mass["zone_mass"].le(0).any():
        raise ValueError(
            "The zone-mass file contains missing, invalid, or non-positive values."
        )

    return mass.reset_index(drop=True)

In [ ]:
def fit_gravity_model(
    df: pd.DataFrame,
    flow_col: str,
    time_col: str,
) -> tuple[dict, np.ndarray]:
    """
    Estimate:

        log(T_ij) = a0
                    + a1 * [log(m_o) + log(m_d)]
                    - a2 * log(d_ij)
    """
    y = np.log(df[flow_col].to_numpy(dtype=float))

    mass_term = (
        np.log(df["m_o"].to_numpy(dtype=float))
        + np.log(df["m_d"].to_numpy(dtype=float))
    )
    negative_log_time = -np.log(
        df[time_col].to_numpy(dtype=float)
    )

    design = np.column_stack([
        np.ones(len(df), dtype=float),
        mass_term,
        negative_log_time,
    ])

    beta, _, _, _ = np.linalg.lstsq(
        design,
        y,
        rcond=None,
    )

    predicted_log_flow = design @ beta
    predicted_flow = np.exp(predicted_log_flow)

    parameters = {
        "a0": float(beta[0]),
        "a1": float(beta[1]),
        "a2": float(beta[2]),
    }

    return parameters, predicted_flow


def apply_ipf(
    df: pd.DataFrame,
    seed_col: str,
    target_col: str,
    max_iter: int = IPF_MAX_ITER,
    tol: float = IPF_TOL,
    eps: float = IPF_EPS,
) -> np.ndarray:
    """Adjust predicted OD flows to the observed origin and destination totals."""
    x = df[seed_col].to_numpy(dtype=float)
    x = np.where(np.isfinite(x) & (x > 0), x, eps)

    origin = df["origin_id"].to_numpy()
    destination = df["destination_id"].to_numpy()

    target_origin = df.groupby("origin_id")[target_col].sum()
    target_destination = df.groupby("destination_id")[target_col].sum()

    for _ in range(max_iter):
        previous = x.copy()

        current_origin = pd.Series(x).groupby(origin).sum()
        origin_factor = (
            target_origin / current_origin.reindex(target_origin.index)
        ).to_dict()
        x = np.array([
            value * origin_factor.get(zone, 1.0)
            for zone, value in zip(origin, x)
        ])

        current_destination = pd.Series(x).groupby(destination).sum()
        destination_factor = (
            target_destination
            / current_destination.reindex(target_destination.index)
        ).to_dict()
        x = np.array([
            value * destination_factor.get(zone, 1.0)
            for zone, value in zip(destination, x)
        ])

        relative_change = (
            np.sum(np.abs(x - previous))
            / (np.sum(np.abs(previous)) + eps)
        )

        if relative_change < tol:
            break

    return x


def cpc_ratio(
    observed: np.ndarray,
    predicted: np.ndarray,
) -> float:
    """Compute CPC as a ratio ranging from 0 to 1."""
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    denominator = observed.sum() + predicted.sum()

    if denominator <= 0:
        return np.nan

    return float(
        2.0 * np.minimum(observed, predicted).sum()
        / denominator
    )


def r2_on_proportions(
    observed: np.ndarray,
    predicted: np.ndarray,
) -> float:
    """Compute R² after converting observed and predicted flows to OD proportions."""
    observed = np.asarray(observed, dtype=float)
    predicted = np.asarray(predicted, dtype=float)

    observed_total = observed.sum()
    predicted_total = predicted.sum()

    if observed_total <= 0 or predicted_total <= 0:
        return np.nan

    observed_share = observed / observed_total
    predicted_share = predicted / predicted_total

    ss_res = np.sum((observed_share - predicted_share) ** 2)
    ss_tot = np.sum(
        (observed_share - observed_share.mean()) ** 2
    )

    if ss_tot == 0:
        return 0.0

    return float(1.0 - ss_res / ss_tot)

## 2. Load the HTS, LCS, and zone-mass inputs

In [ ]:
HTS_OD_RAW = read_csv_flexible(HTS_OD_FILE)
LCS_OD_RAW = read_csv_flexible(LCS_OD_FILE)
HTS_TRIP_RAW = read_csv_flexible(HTS_TRIP_FILE)
LCS_TRIP_RAW = read_csv_flexible(LCS_TRIP_FILE)
ZONE_MASS = load_zone_mass(ZONE_MASS_FILE)

print(f"HTS OD rows: {len(HTS_OD_RAW):,}")
print(f"LCS OD rows: {len(LCS_OD_RAW):,}")
print(f"HTS processed trip rows: {len(HTS_TRIP_RAW):,}")
print(f"LCS processed trip rows: {len(LCS_TRIP_RAW):,}")
print(f"Zone-mass rows: {len(ZONE_MASS):,}")

## 3. Construct the common interzonal OD modeling data

In [ ]:
HTS_FLOW = prepare_od_flow_table(
    HTS_OD_RAW,
    flow_name="hts_flow",
)
LCS_FLOW = prepare_od_flow_table(
    LCS_OD_RAW,
    flow_name="lcs_flow",
)

HTS_TIME = aggregate_od_travel_time(
    HTS_TRIP_RAW,
    output_col="hts_travel_time_min",
)
LCS_TIME = aggregate_od_travel_time(
    LCS_TRIP_RAW,
    output_col="lcs_travel_time_min",
)

# Retain interzonal OD pairs with non-zero flows in both datasets.
COMMON_OD = HTS_FLOW.merge(
    LCS_FLOW,
    on=["origin_id", "destination_id"],
    how="inner",
    validate="one_to_one",
)

COMMON_OD = COMMON_OD.loc[
    COMMON_OD["origin_id"].ne(COMMON_OD["destination_id"])
    & COMMON_OD["hts_flow"].gt(0)
    & COMMON_OD["lcs_flow"].gt(0)
].copy()

COMMON_OD = COMMON_OD.merge(
    HTS_TIME,
    on=["origin_id", "destination_id"],
    how="left",
    validate="one_to_one",
)
COMMON_OD = COMMON_OD.merge(
    LCS_TIME,
    on=["origin_id", "destination_id"],
    how="left",
    validate="one_to_one",
)

origin_mass = ZONE_MASS.rename(columns={
    "zone_id": "origin_id",
    "zone_mass": "m_o",
})
destination_mass = ZONE_MASS.rename(columns={
    "zone_id": "destination_id",
    "zone_mass": "m_d",
})

COMMON_OD = COMMON_OD.merge(
    origin_mass,
    on="origin_id",
    how="left",
    validate="many_to_one",
)
COMMON_OD = COMMON_OD.merge(
    destination_mass,
    on="destination_id",
    how="left",
    validate="many_to_one",
)

if COMMON_OD[["m_o", "m_d"]].isna().any().any():
    raise ValueError(
        "Some common OD pairs could not be matched to the zone-mass file."
    )

# The historical result-producing implementation uses the corresponding
# OD-level flow-weighted mean travel time for each dataset.
HTS_MODEL_DATA = COMMON_OD.loc[
    COMMON_OD["hts_travel_time_min"].notna()
    & COMMON_OD["hts_travel_time_min"].gt(0)
    & COMMON_OD["m_o"].gt(0)
    & COMMON_OD["m_d"].gt(0)
].copy()

LCS_MODEL_DATA = COMMON_OD.loc[
    COMMON_OD["lcs_travel_time_min"].notna()
    & COMMON_OD["lcs_travel_time_min"].gt(0)
    & COMMON_OD["m_o"].gt(0)
    & COMMON_OD["m_d"].gt(0)
].copy()

INPUT_SUMMARY = pd.DataFrame([
    {
        "dataset": "Common HTS-LCS OD support",
        "n_od_pairs": len(COMMON_OD),
        "n_origin_zones": COMMON_OD["origin_id"].nunique(),
        "n_destination_zones": COMMON_OD["destination_id"].nunique(),
    },
    {
        "dataset": "Expanded HTS model input",
        "n_od_pairs": len(HTS_MODEL_DATA),
        "n_origin_zones": HTS_MODEL_DATA["origin_id"].nunique(),
        "n_destination_zones": HTS_MODEL_DATA["destination_id"].nunique(),
    },
    {
        "dataset": "Pre-expanded LCS model input",
        "n_od_pairs": len(LCS_MODEL_DATA),
        "n_origin_zones": LCS_MODEL_DATA["origin_id"].nunique(),
        "n_destination_zones": LCS_MODEL_DATA["destination_id"].nunique(),
    },
])

display(INPUT_SUMMARY)

## 4. Estimate the gravity models and apply IPF

In [ ]:
MODEL_SPECS = [
    {
        "dataset": "Expanded HTS",
        "data": HTS_MODEL_DATA,
        "flow_col": "hts_flow",
        "time_col": "hts_travel_time_min",
    },
    {
        "dataset": "Pre-expanded LCS",
        "data": LCS_MODEL_DATA,
        "flow_col": "lcs_flow",
        "time_col": "lcs_travel_time_min",
    },
]

parameter_rows = []
performance_rows = []
prediction_tables = []

for spec in MODEL_SPECS:
    dataset_name = spec["dataset"]
    model_data = spec["data"].copy()
    flow_col = spec["flow_col"]
    time_col = spec["time_col"]

    parameters, predicted_flow = fit_gravity_model(
        model_data,
        flow_col=flow_col,
        time_col=time_col,
    )

    model_data["observed_flow"] = model_data[flow_col]
    model_data["travel_time_min"] = model_data[time_col]
    model_data["predicted_flow"] = predicted_flow
    model_data["predicted_flow_ipf"] = apply_ipf(
        model_data,
        seed_col="predicted_flow",
        target_col="observed_flow",
    )

    observed = model_data["observed_flow"].to_numpy(dtype=float)
    predicted = model_data["predicted_flow"].to_numpy(dtype=float)
    predicted_ipf = model_data["predicted_flow_ipf"].to_numpy(dtype=float)

    parameter_rows.append({
        "dataset": dataset_name,
        "a0": parameters["a0"],
        "a1": parameters["a1"],
        "a2": parameters["a2"],
        "n_od_pairs": len(model_data),
    })

    performance_rows.extend([
        {
            "dataset": dataset_name,
            "calibration": "Unconstrained",
            "CPC": cpc_ratio(observed, predicted),
            "R2": r2_on_proportions(observed, predicted),
        },
        {
            "dataset": dataset_name,
            "calibration": "IPF",
            "CPC": cpc_ratio(observed, predicted_ipf),
            "R2": r2_on_proportions(observed, predicted_ipf),
        },
    ])

    prediction_table = model_data[[
        "origin_id",
        "destination_id",
        "observed_flow",
        "travel_time_min",
        "m_o",
        "m_d",
        "predicted_flow",
        "predicted_flow_ipf",
    ]].copy()
    prediction_table.insert(0, "dataset", dataset_name)
    prediction_tables.append(prediction_table)

GRAVITY_PARAMETERS = pd.DataFrame(parameter_rows)
GRAVITY_PERFORMANCE = pd.DataFrame(performance_rows)
GRAVITY_PREDICTIONS = pd.concat(
    prediction_tables,
    ignore_index=True,
)

print("Gravity model estimation completed")

## 5. Review and save the outputs

In [ ]:
display(GRAVITY_PARAMETERS.round(4))
display(GRAVITY_PERFORMANCE.round(4))

In [ ]:
INPUT_SUMMARY.to_csv(
    GRAVITY_INPUT_SUMMARY_FILE,
    index=False,
    encoding="utf-8-sig",
)
GRAVITY_PARAMETERS.to_csv(
    GRAVITY_PARAMETER_FILE,
    index=False,
    encoding="utf-8-sig",
)
GRAVITY_PERFORMANCE.to_csv(
    GRAVITY_PERFORMANCE_FILE,
    index=False,
    encoding="utf-8-sig",
)
GRAVITY_PREDICTIONS.to_csv(
    GRAVITY_PREDICTION_FILE,
    index=False,
    encoding="utf-8-sig",
)

print("Save complete")
# print(f"Saved: {GRAVITY_INPUT_SUMMARY_FILE}")
# print(f"Saved: {GRAVITY_PARAMETER_FILE}")
# print(f"Saved: {GRAVITY_PERFORMANCE_FILE}")
# print(f"Saved: {GRAVITY_PREDICTION_FILE}")

# Notes

- Internal trips (`origin_id == destination_id`) are excluded.
- Only OD pairs with positive HTS and LCS flows are retained before model estimation.
- The gravity parameters are estimated before IPF; therefore, the parameters are identical across calibration conditions.
- IPF is applied only to the predicted OD flows and matches the observed origin and destination totals for each dataset.
- CPC is stored as a ratio from 0 to 1, not as a percentage multiplied by 100.
- R² is calculated after converting observed and predicted OD flows into proportions, following the result-producing evaluation code.
- `area_o` and `area_d` are not used in the gravity equation and are therefore not included.
- PKL conversion, BMS loading, train/test duplication, plots, SRMSE, auxiliary error measures, and serialized model objects are not required for reproducing Tables 6 and 7.